# M55 M54-parent compression feasibility

This notebook freezes the selected M54 epoch-100 parent, measures its untouched CUDA model-only baseline, inventories ordinary Conv2d/Linear weight-compression scope, and audits export blockers. **No training or quantization is performed.** Run top-to-bottom on a Colab GPU and stop after the final gate.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from collections import deque
import json, os, shlex, shutil, subprocess, sys
MOBILE_REPO=Path('/content/mobile_adas3d')
MONODGP_REPO=Path('/content/MonoDGP_M55')
MONODGP_COMMIT='aa059a18214aebf644510e7f0793971b403f9d14'
DRIVE_DATASET_ROOT=Path('/content/drive/MyDrive/datasets/kitti')
LOCAL_DATASET_ROOT=Path('/content/kitti')
SPLIT_DIR=Path('/content/drive/MyDrive/mobile_adas3d_splits/kitti_chen')
DATASET_ROOT=Path('/content/monodgp_kitti_m55')
M54_ROOT=Path('/content/drive/MyDrive/mobile_adas3d_outputs/challengers/monodgp_m54')
M54_MANIFEST=M54_ROOT/'m54_adaptation_manifest.json'
M54_SWEEP_DIR=M54_ROOT/'product_checkpoint_sweep'
M54_SELECTION=M54_SWEEP_DIR/'m54_product_selection.json'
M54_SWEEP=M54_SWEEP_DIR/'m54_product_checkpoint_sweep.csv'
OUTPUT_ROOT=Path('/content/drive/MyDrive/mobile_adas3d_outputs/compression/monodgp_m55_feasibility')
LOG_DIR=OUTPUT_ROOT/'colab_logs'
def run(command,cwd=None,env=None):
    command=[str(x) for x in command]; print('+',shlex.join(command),flush=True)
    merged=os.environ.copy(); merged.update(env or {})
    result=subprocess.run(command,cwd=cwd,env=merged)
    if result.returncode: raise RuntimeError(f'Exit {result.returncode}: {shlex.join(command)}')
def run_logged(command,cwd,log_path,env=None):
    command=[str(x) for x in command]; print('+',shlex.join(command),flush=True)
    log_path=Path(log_path); log_path.parent.mkdir(parents=True,exist_ok=True)
    merged=os.environ.copy(); merged.update(env or {})
    tail=deque(maxlen=80)
    with log_path.open('w',encoding='utf-8') as log:
        process=subprocess.Popen(command,cwd=cwd,env=merged,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in process.stdout:
            print(line,end='',flush=True); log.write(line); tail.append(line.rstrip())
        code=process.wait()
    if code: raise RuntimeError(f'Exit {code}; full log={log_path}\n'+'\n'.join(tail))
    return log_path
OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
run(['nvidia-smi'])

In [ ]:
# Fetch exact sources, apply the audited compatibility/taxonomy patches, and build the CUDA extension.
if not MOBILE_REPO.exists():
    run(['git','clone','https://github.com/Ali-RT/mobile_adas3d.git',MOBILE_REPO])
else:
    run(['git','pull','--ff-only'],cwd=MOBILE_REPO)
if not MONODGP_REPO.exists():
    run(['git','clone','https://github.com/PuFanqi23/MonoDGP.git',MONODGP_REPO])
run(['git','fetch','--all'],cwd=MONODGP_REPO)
run(['git','checkout',MONODGP_COMMIT],cwd=MONODGP_REPO)
run([sys.executable,'-m','pip','install','-q','pyyaml','scipy','opencv-python-headless','numba','scikit-image','scikit-learn','tqdm','ninja','pandas'])
run([sys.executable,'scripts/patch_monodgp_colab_compat.py','--monodgp-repo',MONODGP_REPO],cwd=MOBILE_REPO)
run([sys.executable,'scripts/patch_monodgp_m54_training.py','--monodgp-repo',MONODGP_REPO],cwd=MOBILE_REPO)
changed=set(subprocess.run(['git','diff','--name-only'],cwd=MONODGP_REPO,check=True,capture_output=True,text=True).stdout.splitlines())
expected={
    'lib/datasets/kitti/kitti_dataset.py',
    'lib/helpers/save_helper.py',
    'lib/helpers/trainer_helper.py',
    'lib/models/monodgp/ops/modules/ms_deform_attn.py',
    'lib/models/monodgp/ops/setup.py',
    'lib/models/monodgp/ops/src/cuda/ms_deform_attn_cuda.cu',
    'tools/train_val.py',
}
if changed != expected: raise RuntimeError(f'Unexpected patched source set: {changed}')
ops=MONODGP_REPO/'lib/models/monodgp/ops'
shutil.rmtree(ops/'build',ignore_errors=True)
run([sys.executable,'setup.py','build','install'],cwd=ops,env={'MAX_JOBS':'2'})
run([sys.executable,'-c','import torch, MultiScaleDeformableAttention; print(torch.__version__,torch.version.cuda,torch.cuda.get_device_name(0))'],cwd=MONODGP_REPO)

In [ ]:
# Create a canonical Chen-split KITTI view without copying images.
def resolve(root,names):
    for name in names:
        path=root/name
        if path.is_dir(): return path
sources={
    key:resolve(LOCAL_DATASET_ROOT,names) or resolve(DRIVE_DATASET_ROOT,names)
    for key,names in {
        'image_2':['training/image_2','training/image_02'],
        'label_2':['training/label_2','training/label_02'],
        'calib':['training/calib'],
    }.items()
}
if any(path is None for path in sources.values()): raise FileNotFoundError(sources)
(DATASET_ROOT/'training').mkdir(parents=True,exist_ok=True)
(DATASET_ROOT/'ImageSets').mkdir(parents=True,exist_ok=True)
for name,target in sources.items():
    link=DATASET_ROOT/'training'/name
    if link.is_symlink() and link.resolve()==target.resolve(): continue
    if link.exists() or link.is_symlink(): raise RuntimeError(f'Refusing to replace {link}')
    link.symlink_to(target,target_is_directory=True)
for split in ('train','val'):
    shutil.copy2(SPLIT_DIR/f'{split}.txt',DATASET_ROOT/'ImageSets'/f'{split}.txt')
assert len((DATASET_ROOT/'ImageSets/train.txt').read_text().splitlines())==3712
assert len((DATASET_ROOT/'ImageSets/val.txt').read_text().splitlines())==3769
for required in (M54_MANIFEST,M54_SELECTION,M54_SWEEP):
    if not required.is_file(): raise FileNotFoundError(f'M54 evidence missing: {required}')

In [ ]:
# Freeze M54 provenance, preservation denominators, and the deterministic M55 profile config.
PREPARE_LOG=run_logged([
    sys.executable,'-u','scripts/prepare_monodgp_m55_feasibility.py',
    '--monodgp-repo',MONODGP_REPO,
    '--dataset-root',DATASET_ROOT,
    '--m54-manifest',M54_MANIFEST,
    '--m54-selection',M54_SELECTION,
    '--m54-sweep',M54_SWEEP,
    '--output-root',OUTPUT_ROOT,
],MOBILE_REPO,LOG_DIR/'m55_prepare.log')
MANIFEST=OUTPUT_ROOT/'m55_feasibility_manifest.json'
manifest=json.loads(MANIFEST.read_text())
assert manifest['parent_epoch']==100
assert manifest['parent_checkpoint_sha256']=='8e79f3921d96e1de70cbb4219245e3fcc3fa1fb67ae675468b4ebca90e579847'
assert manifest['training_performed'] is False and manifest['compression_performed'] is False
print(json.dumps(manifest['preservation_gates'],indent=2))

In [ ]:
# Real CUDA parent profile: one fixed validation image, 5 warmups, 100 timed model-only predictions.
PROFILE_LOG=run_logged([
    sys.executable,'-u','scripts/profile_monodgp_m55_baseline.py',
    '--monodgp-repo',MONODGP_REPO,
    '--manifest',MANIFEST,
    '--output-dir',OUTPUT_ROOT,
],MOBILE_REPO,LOG_DIR/'m55_native_profile.log')
PROFILE=OUTPUT_ROOT/'m55_native_baseline_profile.json'
AUDIT=OUTPUT_ROOT/'m55_operator_export_audit.json'
LATENCY=OUTPUT_ROOT/'m55_native_baseline_latency.csv'
profile=json.loads(PROFILE.read_text()); audit=json.loads(AUDIT.read_text())
print('GPU:',profile['device']['name'])
print('Parameters:',profile['parameter_inventory']['total_parameters'])
print('Latency:',json.dumps(profile['latency'],indent=2))
print('Ordinary-weight coverage:',audit['eligible_parameter_fraction'])
print('Direct Core ML ready:',audit['direct_coreml_export_ready'])

In [ ]:
# Fail-closed feasibility decision. A pass authorizes only the separate M56 offline experiment.
GATE=OUTPUT_ROOT/'m55_feasibility_gate.json'
FINAL_LOG=run_logged([
    sys.executable,'-u','scripts/finalize_monodgp_m55_feasibility.py',
    '--manifest',MANIFEST,
    '--profile',PROFILE,
    '--operator-audit',AUDIT,
    '--latency-csv',LATENCY,
    '--output',GATE,
],MOBILE_REPO,LOG_DIR/'m55_finalize.log')
gate=json.loads(GATE.read_text())
assert gate['all_feasibility_gates_passed']
assert gate['offline_weight_compression_authorized']
assert gate['direct_coreml_conversion_authorized'] is False
assert gate['product_safety_qualified'] is False
print(json.dumps(gate,indent=2))

## Stop point

Stop here. Return `m55_feasibility_gate.json`, `m55_native_baseline_profile.json`, and `m55_operator_export_audit.json`. Do not start M56 quantization until these results are reviewed. The expected M55 conclusion is that offline weight compression may be testable while direct Core ML conversion remains blocked by custom deformable attention.